In [2]:
# 데이터 처리
import pandas as pd
import numpy as np
# URL 분석 / 문자열 패턴 정리
from urllib.parse import urlparse
import re
# 머신러닝 데이터 분리
from sklearn.model_selection import train_test_split
# 모델: 랜덤 포레스트
from sklearn.ensemble import RandomForestClassifier
# 성능 평가
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 데이터셋 불러오기
import kagglehub
import os

In [3]:
path = kagglehub.dataset_download("sid321axn/malicious-urls-dataset")

print("Path to dataset files:", path)
print(os.listdir(path))

df = pd.read_csv(os.path.join(path, "malicious_phish.csv"))
df.head()

100%|██████████| 16.9M/16.9M [00:00<00:00, 145MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sid321axn/malicious-urls-dataset/versions/1
['malicious_phish.csv']


,url,type
0,br-icloud.com.br,phishing
1,mp3raid.com/music/krizz_kaliko.html,benign
2,bopsecrets.org/rexroth/cr/1.htm,benign
3,http://www.garage-pirenne.be/index.php?option=...,defacement
4,http://adventure-nicaragua.net/index.php?optio...,defacement


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   url     651191 non-null  object
 1   type    651191 non-null  object
dtypes: object(2)
memory usage: 9.9+ MB


In [5]:
df["type"].value_counts()

,count
type,
benign,428103
defacement,96457
phishing,94111
malware,32520


In [6]:
# 라벨 변환
df["label"] = df["type"].apply(lambda x: 0 if x == "benign" else 1)
df[["url", "type", "label"]].head() # 5개 출력

,url,type,label
0,br-icloud.com.br,phishing,1
1,mp3raid.com/music/krizz_kaliko.html,benign,0
2,bopsecrets.org/rexroth/cr/1.htm,benign,0
3,http://www.garage-pirenne.be/index.php?option=...,defacement,1
4,http://adventure-nicaragua.net/index.php?optio...,defacement,1


In [7]:
# URL 문자열의 특징들을 숫자로 나타내는 함수 구현
def normalize_for_parse(url):
    url = str(url)

    # urlparse가 도메인을 제대로 인식하도록 scheme이 없으면 임시로 http:// 추가
    if not url.startswith(("http://", "https://")):
        return "http://" + url

    return url

def has_ip_address(url):
    pattern = r'(\d{1,3}\.){3}\d{1,3}'
    return 1 if re.search(pattern, str(url)) else 0

def extract_features(url):
    url = str(url)
    parsed_url = normalize_for_parse(url)

    try:
        parsed = urlparse(parsed_url)
        domain = parsed.netloc
        path = parsed.path
        query = parsed.query
    except ValueError:
        # urlparse가 처리하지 못하는 이상한 URL은 기본값으로 처리
        domain = ""
        path = url
        query = ""

    features = {
        "url_length": len(url),
        "domain_length": len(domain),
        "path_length": len(path),
        "count_dot": url.count("."),
        "count_hyphen": url.count("-"),
        "count_at": url.count("@"),
        "count_question": url.count("?"),
        "count_equal": url.count("="),
        "count_digits": sum(c.isdigit() for c in url),
        "uses_https": 1 if url.startswith("https://") else 0,
        "has_ip": has_ip_address(url),
    }

    return features

In [8]:
# URL 전체를 숫자 특징으로 바꾸기
features = df["url"].apply(extract_features)

# 숫자들의 특징을 나타내는 표 생성
X = pd.DataFrame(features.tolist())
X.head()

,url_length,domain_length,path_length,count_dot,count_hyphen,count_at,count_question,count_equal,count_digits,uses_https,has_ip
0,16,16,0,2,1,0,0,0,0,0,0
1,35,11,24,2,0,0,0,0,1,0,0
2,31,14,17,2,0,0,0,0,1,0,0
3,88,21,10,3,1,0,1,4,7,0,0
4,235,23,10,2,1,0,1,3,22,0,0


In [9]:
# 정답 라벨 생성
y = df["label"]
y.head()

,label
0,1
1,0
2,0
3,1
4,1


In [10]:
# X 확인
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   url_length      651191 non-null  int64
 1   domain_length   651191 non-null  int64
 2   path_length     651191 non-null  int64
 3   count_dot       651191 non-null  int64
 4   count_hyphen    651191 non-null  int64
 5   count_at        651191 non-null  int64
 6   count_question  651191 non-null  int64
 7   count_equal     651191 non-null  int64
 8   count_digits    651191 non-null  int64
 9   uses_https      651191 non-null  int64
 10  has_ip          651191 non-null  int64
dtypes: int64(11)
memory usage: 54.7 MB


In [11]:
X.shape

(651191, 11)

In [12]:
# 특징 추출
df_sample = df.sample(651191, random_state=42)

features = df_sample["url"].apply(extract_features)
X = pd.DataFrame(features.tolist())

y = df_sample["label"]

X.head()

,url_length,domain_length,path_length,count_dot,count_hyphen,count_at,count_question,count_equal,count_digits,uses_https,has_ip
0,38,13,18,4,0,0,0,0,11,0,1
1,54,40,14,2,1,0,0,0,0,0,0
2,26,14,12,2,0,0,0,0,0,0,0
3,49,15,27,3,0,0,0,0,3,0,0
4,121,18,103,2,5,0,0,0,7,0,0


In [13]:
# 학습을 잘 했는지 확인
print(X.shape)
print(y.shape)

(651191, 11)
(651191,)


In [14]:
# RandomForest 모델 학습

# 학습용, 테스트용 분류: 학습용 80%, 테스트용 20%
X_train, X_test, y_train, y_test = train_test_split (
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 잘 분류되었는지 확인
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(520952, 11)
(130239, 11)
(520952,)
(130239,)


In [15]:
# Random Forest 모델 만들고 학습
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=42)

In [16]:
# 테스트 데이터 예측
y_pred = model.predict(X_test)

In [17]:
# 성능 평가 출력
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9355031902886233

Confusion Matrix:
[[82494  3127]
 [ 5273 39345]]

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.96      0.95     85621
           1       0.93      0.88      0.90     44618

    accuracy                           0.94    130239
   macro avg       0.93      0.92      0.93    130239
weighted avg       0.94      0.94      0.94    130239



In [18]:
# 악성 URL 판별 함수 구현
def predict_url(url):
    # 1. 입력된 URL을 숫자 특징으로 변환
    feature = extract_features(url)
    feature_df = pd.DataFrame([feature])

    # 2. 학습된 모델로 예측
    pred = model.predict(feature_df)[0]

    # 3. 예측 확률 계산
    prob = model.predict_proba(feature_df)[0]

    # 4. 결과 출력
    if pred == 0:
        print("예측 결과: 정상 URL")
        print(f"정상 확률: {prob[0]:.4f}")
        print(f"악성 확률: {prob[1]:.4f}")
    else:
        print("예측 결과: 악성 URL")
        print(f"정상 확률: {prob[0]:.4f}")
        print(f"악성 확률: {prob[1]:.4f}")

In [19]:
predict_url("https://www.naver.com")

예측 결과: 악성 URL
정상 확률: 0.0000
악성 확률: 1.0000


In [20]:
predict_url("http://192.168.0.1/login/verify-account")

예측 결과: 악성 URL
정상 확률: 0.0300
악성 확률: 0.9700


In [21]:
predict_url("http://secure-login-account-verification.com/update")

예측 결과: 정상 URL
정상 확률: 0.5800
악성 확률: 0.4200
